In [1]:
!pip install -q avalanche-lib==0.4

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 894.6/894.6 kB 8.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 532.4/532.4 kB 13.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 23.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 806.1/806.1 kB 27.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 452.8/452.8 kB 26.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.3/115.3 kB 8.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.6/190.6 kB 19.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.0/254.0 kB 17.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 6.3 MB/s eta 0:00:00


In [2]:
import torch
from torch import nn
from torch.optim import SGD
from torch.nn import CrossEntropyLoss
import torchvision
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from avalanche.benchmarks.generators import nc_benchmark, ni_benchmark, dataset_benchmark
from avalanche.evaluation.metrics import forgetting_metrics, accuracy_metrics, loss_metrics, timing_metrics, cpu_usage_metrics, confusion_matrix_metrics, disk_usage_metrics
from avalanche.models import SimpleCNN, SimpleMLP, SimpleMLP_TinyImageNet, MobilenetV1, IncrementalClassifier, MultiHeadClassifier
from avalanche.logging import InteractiveLogger, TextLogger, TensorboardLogger
from avalanche.training.plugins import EvaluationPlugin
from avalanche.training.supervised import Naive


from pathlib import Path
import zipfile
import os
import shutil
import pandas as pd
import numpy as np

In [3]:
HX_TRAINING_ORIGIN_DIR_PATH = Path('./drive/MyDrive/Colab Notebooks/hx_training_classify')

In [4]:
DATA_DIR_PATH = Path('./data')
if os.path.exists(DATA_DIR_PATH):
  print(f"{DATA_DIR_PATH} directory already exists")
else:
   DATA_DIR_PATH.mkdir(parents=True, exist_ok=True)

In [5]:
#Copy hx_training_directory to data_dir
HX_TRAINING_DEST_DIR_PATH = os.path.join(DATA_DIR_PATH,"hx_training")
shutil.copytree(HX_TRAINING_ORIGIN_DIR_PATH, HX_TRAINING_DEST_DIR_PATH)

'data/hx_training'

In [6]:
#Transform CSV file to a DataFrame and a Dictionary
ANNOTATIONS_PATH = Path('./data/hx_training/FSW dataset A_annotations.csv')
df_annotations = pd.read_csv(ANNOTATIONS_PATH, sep=";")

In [7]:
df_annotations

,#,begin,end,Temperatur,Grat,Nahtinneres,temp_label,grat_label,inner_label,filename,img_path,Weld Seam Nr.,Group
0,0,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-25.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-2...,1,1
1,1,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-14.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-1...,1,1
2,2,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-37.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-3...,1,1
3,3,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-57.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-5...,1,1
4,4,321,3573,Temperatur OK,Grat OK,Nahtinneres OK,0,0,0,region-2-18.JPG,../data/raw/hx_fsw/images/1/cutouts/region-2-1...,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8319,8455,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-52.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7
8320,8456,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-36.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7
8321,8457,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-31.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7
8322,8458,0,3617,Temperatur zu heiss,Grat zu stark,Nahtinneres OK,2,1,0,region-1-44.JPG,../data/raw/hx_fsw/images/114/cutouts/region-1...,114,7


In [8]:
df_dictionary = df_annotations.to_dict()

In [9]:
#Unzip image folder
ZIPFILE_PATH = os.path.join(HX_TRAINING_DEST_DIR_PATH, "hx_training_classify.zip")
with zipfile.ZipFile(ZIPFILE_PATH, 'r') as zip_ref:
  zip_ref.extractall(DATA_DIR_PATH)

In [10]:
def creationLabelFolders(CREATION_PATH, groups, labels, dir_name):
  MAIN_DIR_PATH = os.path.join(CREATION_PATH, dir_name)
  MAIN_DIR_PATH = Path(MAIN_DIR_PATH)

  path_dictionary = {}
  if os.path.exists(MAIN_DIR_PATH):
    print(f"{dir_name} folder already exists")
  else:
    MAIN_DIR_PATH.mkdir(parents=True, exist_ok=True)

  for group in groups:
    GROUP_DIR_PATH = Path(os.path.join(MAIN_DIR_PATH, f"group_{group}"))
    if os.path.exists(GROUP_DIR_PATH):
      print(f"{GROUP_DIR_PATH} already exists")
    else:
      GROUP_DIR_PATH.mkdir(parents=True, exist_ok=True)
    path_dictionary[f"GROUP_{group}_PATH"] = GROUP_DIR_PATH
    for label in labels:
      LABEL_DIR_PATH = Path(os.path.join(GROUP_DIR_PATH, label))
      if os.path.exists(LABEL_DIR_PATH):
        print(f"{LABEL_DIR_PATH} already exists")
      else:
        LABEL_DIR_PATH.mkdir(parents=True, exist_ok=True)
      #path_dictionary[f"{label}_PATH"] = LABEL_DIR_PATH
  return path_dictionary

In [11]:
#Create label folders
unique_grat_keys = set(df_dictionary['Grat'].values())
unique_grat_keys_list = list(unique_grat_keys)
print(unique_grat_keys_list)

groups_keys = set(df_dictionary['Group'].values())
groups_keys_list = list(groups_keys)
print(groups_keys_list)

grat_path_dic = creationLabelFolders(DATA_DIR_PATH, groups_keys_list, unique_grat_keys_list, "Grat")

['Grat OK', 'Grat zu stark']
[1, 2, 3, 4, 5, 6, 7]


In [12]:
grat_path_dic

{'GROUP_1_PATH': PosixPath('data/Grat/group_1'),
 'GROUP_2_PATH': PosixPath('data/Grat/group_2'),
 'GROUP_3_PATH': PosixPath('data/Grat/group_3'),
 'GROUP_4_PATH': PosixPath('data/Grat/group_4'),
 'GROUP_5_PATH': PosixPath('data/Grat/group_5'),
 'GROUP_6_PATH': PosixPath('data/Grat/group_6'),
 'GROUP_7_PATH': PosixPath('data/Grat/group_7')}

In [13]:
grat_path_list = list(grat_path_dic.values())
grat_path_list

[PosixPath('data/Grat/group_1'),
 PosixPath('data/Grat/group_2'),
 PosixPath('data/Grat/group_3'),
 PosixPath('data/Grat/group_4'),
 PosixPath('data/Grat/group_5'),
 PosixPath('data/Grat/group_6'),
 PosixPath('data/Grat/group_7')]

In [14]:
for key, value in df_dictionary['filename'].items():
    weld_seam_pos = df_dictionary['Weld Seam Nr.'][key]
    group_pos = df_dictionary['Group'][key]
    filename_pos = df_dictionary['filename'][key]
    file_path = Path(f"./data/hx_training_classify/{weld_seam_pos}/cutouts/{filename_pos}")
    label_pos = df_dictionary['Grat'][key]
    if label_pos == unique_grat_keys_list[0]:
      new_file_path = Path(f"./data/Grat/group_{df_dictionary['Group'][key]}/{unique_grat_keys_list[0]}/{filename_pos}_{weld_seam_pos}.jpg")
      #print(new_file_path)
      shutil.copy2(file_path,new_file_path)
    else:
      new_file_path = Path(f"./data/Grat/group_{df_dictionary['Group'][key]}/{unique_grat_keys_list[1]}/{filename_pos}_{weld_seam_pos}.jpg")
      #print(new_file_path)
      shutil.copy2(file_path,new_file_path)

In [15]:
data_transform = transforms.Compose([
    transforms.ToTensor()
])

In [16]:
group_1_dataset = datasets.ImageFolder(root=grat_path_list[0],
                                       transform = None,
                                       target_transform = None)
group_2_dataset = datasets.ImageFolder(root=grat_path_list[1],
                                       transform = None,
                                       target_transform = None)
group_3_dataset = datasets.ImageFolder(root=grat_path_list[2],
                                       transform = None,
                                       target_transform = None)
group_4_dataset = datasets.ImageFolder(root=grat_path_list[3],
                                       transform = None,
                                       target_transform = None)
group_5_dataset = datasets.ImageFolder(root=grat_path_list[4],
                                       transform = None,
                                       target_transform = None)
group_6_dataset = datasets.ImageFolder(root=grat_path_list[5],
                                       transform = None,
                                       target_transform = None)
group_7_dataset = datasets.ImageFolder(root=grat_path_list[6],
                                       transform = None,
                                       target_transform = None)
dataset_arr = [group_1_dataset, group_2_dataset, group_3_dataset,group_4_dataset,group_5_dataset,group_6_dataset,group_7_dataset]

In [17]:
len(dataset_arr[0])

950

In [18]:
#Split datasets
splitted_datasets={"group_1_dataset":{},"group_2_dataset":{},"group_3_dataset":{},"group_4_dataset":{},"group_5_dataset":{},"group_6_dataset":{},"group_7_dataset":{}}

for i in range(0,7):
  train_size = int(0.8 * len(dataset_arr[i]))
  test_size = len(dataset_arr[i]) - train_size
  #j = 1
  splitted_datasets[f"group_{i+1}_dataset"]["train_dataset"], splitted_datasets[f"group_{i+1}_dataset"]["test_dataset"] = torch.utils.data.random_split(dataset_arr[i], [train_size, test_size])
  #j = j+1
  #print(j)

splitted_datasets

{'group_1_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x79392934dc00>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7937ed62f400>},
 'group_2_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x7937ed62ff70>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7937ed62f310>},
 'group_3_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x7937ed62e890>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7937ed62ead0>},
 'group_4_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x7937ed62fd90>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7937ed62f700>},
 'group_5_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x7937ed62f6d0>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7937ed62fa30>},
 'group_6_dataset': {'train_dataset': <torch.utils.data.dataset.Subset at 0x7937ec4ec040>,
  'test_dataset': <torch.utils.data.dataset.Subset at 0x7937ec4ec0a0>},
 'group_7_dataset': {'

In [19]:
i = 1
print(len(splitted_datasets[f"group_{i}_dataset"]["train_dataset"]))

760


In [20]:
i = 1
len(splitted_datasets[f"group_{i}_dataset"]["train_dataset"])

760

In [21]:
for i in range(1,8):
  print(f"group_{i}\n")
  len_train = len(splitted_datasets[f"group_{i}_dataset"]["train_dataset"])
  len_test = len(splitted_datasets[f"group_{i}_dataset"]["test_dataset"])
  print(f"train_dataset lenght: {len_train}")
  print(f"test_dataset lenght: {len_test}\n ====================")

group_1

train_dataset lenght: 760
test_dataset lenght: 190
group_2

train_dataset lenght: 772
test_dataset lenght: 194
group_3

train_dataset lenght: 686
test_dataset lenght: 172
group_4

train_dataset lenght: 1225
test_dataset lenght: 307
group_5

train_dataset lenght: 852
test_dataset lenght: 214
group_6

train_dataset lenght: 1217
test_dataset lenght: 305
group_7

train_dataset lenght: 1144
test_dataset lenght: 286


In [22]:
splitted_datasets['group_1_dataset']["train_dataset"]

##Benchmarks

In [23]:
scenario_test_nc = nc_benchmark(train_dataset = splitted_datasets['group_1_dataset']["train_dataset"],
                             test_dataset = splitted_datasets['group_1_dataset']["test_dataset"],
                             n_experiences=2,
                             shuffle=True,
                             seed=42,
                             task_labels=False)
train_stream_test_nc = scenario_test_nc.train_stream

for experience in train_stream_test_nc:
  t = experience.task_label
  exp_id = experience.current_experience
  training_dataset = experience.dataset
  print('Task {} batch {} --> train'.format(t, exp_id))
  print("This batch contains", len(training_dataset), "patterns")
  print('Classes in this task:', experience.classes_in_this_experience)

Task 0 batch 0 --> train
This batch contains 546 patterns
Classes in this task: [0]
Task 0 batch 1 --> train
This batch contains 214 patterns
Classes in this task: [1]


In [24]:
scenario_test_nc

This one is the easiest way to do a scenario for stream data

In [25]:
scenario_test_dataset = dataset_benchmark(train_datasets=(splitted_datasets['group_1_dataset']["train_dataset"],splitted_datasets['group_2_dataset']["train_dataset"]),
                             test_datasets = (splitted_datasets['group_1_dataset']["test_dataset"], splitted_datasets['group_2_dataset']["test_dataset"]))

In [26]:
train_datasets_scenario = []
test_datasets_scenario = []

for i in range(1, len(splitted_datasets) + 1):
  train_datasets_scenario.append(splitted_datasets[f"group_{i}_dataset"]["train_dataset"])
  test_datasets_scenario.append(splitted_datasets[f"group_{i}_dataset"]["test_dataset"])

In [27]:
train_datasets_scenario, test_datasets_scenario

([<torch.utils.data.dataset.Subset at 0x79392934dc00>,
  <torch.utils.data.dataset.Subset at 0x7937ec4ec160>])

In [28]:
train_datasets_scenario[0]

In [29]:
scenario_dataset_benchmark = dataset_benchmark(train_datasets=train_datasets_scenario, test_datasets= test_datasets_scenario, train_transform = data_transform, eval_transform=None)

In [30]:
scenario_dataset_benchmark

In [31]:
train_stream = scenario_dataset_benchmark.train_stream
test_stream = scenario_dataset_benchmark.test_stream

In [32]:
for experience in train_stream:
  t = experience.task_label
  exp_id = experience.current_experience
  training_dataset = experience.dataset
  print('Task {} batch {} --> train'.format(t, exp_id))
  print("This batch contains", len(training_dataset), "patterns")
  print('Classes in this task:', experience.classes_in_this_experience)

Task 0 batch 0 --> train
This batch contains 760 patterns
Classes in this task: [0, 1]
Task 0 batch 1 --> train
This batch contains 772 patterns
Classes in this task: [0, 1]
Task 0 batch 2 --> train
This batch contains 686 patterns
Classes in this task: [0, 1]
Task 0 batch 3 --> train
This batch contains 1225 patterns
Classes in this task: [0, 1]
Task 0 batch 4 --> train
This batch contains 852 patterns
Classes in this task: [0, 1]
Task 0 batch 5 --> train
This batch contains 1217 patterns
Classes in this task: [0, 1]
Task 0 batch 6 --> train
This batch contains 1144 patterns
Classes in this task: [0, 1]


In [33]:
for experience in train_stream:
  t = experience.task_label
  exp_id = experience.current_experience
  training_dataset = experience.dataset
  print('Task {} batch {} --> train'.format(t, exp_id))
  print("This batch contains", len(training_dataset), "patterns")
  print('Classes in this task:', experience.classes_in_this_experience)

Task 0 batch 0 --> train
This batch contains 760 patterns
Classes in this task: [0, 1]
Task 0 batch 1 --> train
This batch contains 772 patterns
Classes in this task: [0, 1]
Task 0 batch 2 --> train
This batch contains 686 patterns
Classes in this task: [0, 1]
Task 0 batch 3 --> train
This batch contains 1225 patterns
Classes in this task: [0, 1]
Task 0 batch 4 --> train
This batch contains 852 patterns
Classes in this task: [0, 1]
Task 0 batch 5 --> train
This batch contains 1217 patterns
Classes in this task: [0, 1]
Task 0 batch 6 --> train
This batch contains 1144 patterns
Classes in this task: [0, 1]


##Training

In [38]:
model = SimpleCNN(num_classes=2)
optimizer = SGD(model.parameters(), lr=0.001, momentum=0.9)
loss_fn = CrossEntropyLoss()
cl_strategy = Naive(model, optimizer, loss_fn, train_mb_size=64, train_epochs=4, eval_mb_size=64)

In [39]:
print(model)

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Dropout(p=0.25, inplace=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU(inplace=True)
    (8): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
    (9): ReLU(inplace=True)
    (10): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (11): Dropout(p=0.25, inplace=False)
    (12): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1))
    (13): ReLU(inplace=True)
    (14): AdaptiveMaxPool2d(output_size=1)
    (15): Dropout(p=0.25, inplace=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=2, bias=True)
  )
)


In [40]:
print('Starting experiment...')
results=[]
for experience in train_stream:
    print("Start of experience: ", experience.current_experience)
    print("Current Classes: ", experience.classes_in_this_experience)

    cl_strategy.train(experience)
    print('Training completed')

Starting experiment...
Start of experience:  0
Current Classes:  [0, 1]
-- >> Start of training phase << --
100%|██████████| 12/12 [02:11<00:00, 10.94s/it]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.6912
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.5461
100%|██████████| 12/12 [02:17<00:00, 11.47s/it]
Epoch 1 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.6722
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.6934
100%|██████████| 12/12 [02:20<00:00, 11.72s/it]
Epoch 2 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.6529
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.7184
100%|██████████| 12/12 [02:16<00:00, 11.34s/it]
Epoch 3 ended.
	Loss_Epoch/train_phase/train_stream/Task000 = 0.6335
	Top1_Acc_Epoch/train_phase/train_stream/Task000 = 0.7184
-- >> End of training phase << --
Training completed
Start of experience:  1
Current Classes:  [0, 1]
-- >> Start of training phase << --
100%|██████████| 13/13 [02:16<00:00, 10.51s/it]
Epoch 